In [ ]:
import os

os.makedirs("../csv", exist_ok=True)
os.makedirs("../svg", exist_ok=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scienceplots
import pandas as pd
import numpy as np
from scipy.stats import theilslopes
from scipy.stats import gmean


In [ ]:
plt.rcParams.update({
    'font.family': 'TeX Gyre Termes',
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
})

In [ ]:
TEXTWIDTH_PT = 426.79137
TEXTHEIGHT_PT = 702.78308
PT_PER_INCH = 72.27

FIG_WIDTH = TEXTWIDTH_PT / PT_PER_INCH
FIG_HEIGHT_1 = TEXTHEIGHT_PT / PT_PER_INCH
FIG_HEIGHT_2 = FIG_HEIGHT_1 / 2
FIG_HEIGHT_3 = FIG_HEIGHT_1 / 3
FIG_HEIGHT_4 = FIG_HEIGHT_1 / 4
FIG_HEIGHT_5 = FIG_HEIGHT_1 / 5
FIG_HEIGHT_6 = FIG_HEIGHT_1 / 6
FIG_HEIGHT_7 = FIG_HEIGHT_1 / 7
FIG_HEIGHT_8 = FIG_HEIGHT_1 / 8

In [ ]:
operacje_pl = {
    'decrypt': 'deszyfrowanie',
    'encrypt': 'szyfrowanie',
    'encrypt_inflight': 'szyfrowanie\nw locie',
}
backendy_pl = {
    'scalar': 'skalarny',
    'sse2': 'SSE2',
    'avx2': 'AVX2',
    'avx512': 'AVX-512',
    'neon': 'NEON',
}

In [ ]:
ciretrion_x86 = pd.read_csv('../data/criterion_x86.csv')
ciretrion_aarch64 = pd.read_csv('../data/criterion_aarch64.csv')
ciretrion = pd.concat([ciretrion_x86, ciretrion_aarch64], ignore_index=True)
ciretrion

In [ ]:
speck = ciretrion[ciretrion['benchmark'] == 'speck']
speck = speck.drop(columns=['benchmark', 'suffix', 'unit', 'throughput_type'])
speck['time_per_iter_ns'] = speck['sample_measured_value'] / speck['iteration_count']

speck['keys_per_sec'] = (
        speck['throughput_num'] / (speck['time_per_iter_ns'] / 1e9)
)
speck

In [ ]:
group_cols = ['backend', 'architecture', 'function', 'version']
results = []

for keys, group in speck.groupby(group_cols):
    x = group['iteration_count'].values
    y = group['sample_measured_value'].values

    slope, intercept, ci_low, ci_high = theilslopes(y, x, 0.95)

    y_pred = slope * x + intercept
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot

    keys_per_iter = group['throughput_num'].iloc[0]
    keys_per_sec = keys_per_iter / (slope / 1e9)

    results.append({
        **dict(zip(group_cols, keys)),
        'time_per_iter_ns': slope,
        'ci_95_low_ns': ci_low,
        'ci_95_high_ns': ci_high,
        'intercept_ns': intercept,
        'r2': r2,
        'keys_per_sec': keys_per_sec,
        'Mkeys_per_sec': keys_per_sec / 1e6,
    })

speck_reg = pd.DataFrame(results)
speck_reg

In [ ]:
print(f"Min R^2: {speck_reg['r2'].min():.6f}")
print(f"Czy wszystkie > 0.99: {(speck_reg['r2'] > 0.99).all()}")

print(speck_reg[speck_reg['r2'] <= 0.99][['backend', 'architecture', 'backend', 'function', 'version', 'r2']])

In [ ]:
version_order = ['32_64', '48_72', '48_96', '64_96', '64_128', '96_96', '96_144', '128_128', '128_192', '128_256']

def make_speck_pivot(df, backend_order):
    pivot = df.pivot_table(
        index=['backend', 'version'],
        columns='function',
        values='Mkeys_per_sec'
    ).round(2).reset_index()
    pivot.columns.name = None
    pivot['backend'] = pd.Categorical(pivot['backend'], categories=backend_order, ordered=True)
    pivot['version'] = pd.Categorical(pivot['version'], categories=version_order, ordered=True)
    return pivot.sort_values(['backend', 'version']).reset_index(drop=True)

speck_pivot_x86 = make_speck_pivot(
    speck_reg[speck_reg['architecture'] == 'x86_64'],
    ['scalar', 'sse2', 'avx2', 'avx512']
)
speck_pivot_aarch64 = make_speck_pivot(
    speck_reg[speck_reg['architecture'] == 'aarch64'],
    ['scalar', 'neon']
)

In [ ]:
speck_pivot_x86

In [ ]:
speck_pivot_aarch64

In [ ]:
for pivot, name in [(speck_pivot_x86, 'x86'), (speck_pivot_aarch64, 'aarch64')]:
    out = pivot.copy()
    out['version'] = out['version'].astype(str).str.replace('_', '/', regex=False)
    out.to_csv(f'../csv/speck_pivot_{name}.csv', index=False)

In [ ]:
def make_speck_pivot_mean(pivot, backend_order):
    mean = pivot.groupby('backend')[['decrypt', 'encrypt', 'encrypt_inflight']].agg(gmean).reset_index()
    mean['backend'] = pd.Categorical(mean['backend'], categories=backend_order, ordered=True)
    mean = mean.sort_values('backend')
    return mean.set_index('backend')[['decrypt', 'encrypt', 'encrypt_inflight']]

speck_pivot_mean_x86 = make_speck_pivot_mean(speck_pivot_x86, ['scalar', 'sse2', 'avx2', 'avx512'])
speck_pivot_mean_aarch64 = make_speck_pivot_mean(speck_pivot_aarch64, ['scalar', 'neon'])

speck_pivot_mean_x86

In [ ]:
def plot_speck_heatmap(pivot_mean, filename):
    data = pivot_mean.T.rename(index=operacje_pl, columns=backendy_pl)

    with plt.style.context(['science', 'no-latex', 'bright']):
        fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT_7), layout='constrained')

        sns.heatmap(
            data,
            ax=ax,
            cmap='RdYlGn',
            center=0,
            annot=True,
            fmt='.2f',
            linewidths=1.0,
            cbar_kws={'label': 'Przepustowość\n[Mkeys/s]', 'shrink': 0.85},
        )

        ax.set_xlabel("Backend")
        ax.set_ylabel('Operacja')

        plt.savefig(f'../svg/{filename}', facecolor='white')
        plt.show()

plot_speck_heatmap(speck_pivot_mean_x86, 'speck_pivot_backend_heatmap_x86.svg')
plot_speck_heatmap(speck_pivot_mean_aarch64, 'speck_pivot_backend_heatmap_aarch64.svg')

In [ ]:
speck_encrypt_inflight = speck_reg[speck_reg['function'] == 'encrypt_inflight']
speck_encrypt_inflight = speck_encrypt_inflight.drop(columns=['function'])
speck_encrypt_inflight

In [ ]:
def make_speck_backend_speedup(df, backend_order):
    version_order = ['32_64', '48_72', '48_96', '64_96', '64_128', '96_96', '96_144', '128_128', '128_192', '128_256']

    scalar_perf = df[df['backend'] == 'scalar'].set_index('version')['keys_per_sec']
    speedup = df[df['backend'] != 'scalar'].copy()
    speedup['speedup'] = (speedup['keys_per_sec'] / speedup['version'].map(scalar_perf)).round(2)
    speedup = speedup.drop(columns=['time_per_iter_ns', 'ci_95_high_ns', 'ci_95_low_ns', 'intercept_ns', 'r2', 'keys_per_sec', 'Mkeys_per_sec'])

    speedup['backend'] = pd.Categorical(speedup['backend'], categories=backend_order, ordered=True)
    speedup['version'] = pd.Categorical(speedup['version'], categories=version_order, ordered=True)

    speedup = speedup.pivot_table(index='version', columns='backend', values='speedup').reset_index()
    speedup.columns.name = None
    speedup['version'] = speedup['version'].astype(str).str.replace('_', '/', regex=False)
    return speedup.set_index('version')

speck_backend_speedup_x86 = make_speck_backend_speedup(
    speck_encrypt_inflight[speck_encrypt_inflight['architecture'] == 'x86_64'],
    ['sse2', 'avx2', 'avx512']
)
speck_backend_speedup_aarch64 = make_speck_backend_speedup(
    speck_encrypt_inflight[speck_encrypt_inflight['architecture'] == 'aarch64'],
    ['neon']
)

# speck_backend_speedup_x86

speck_backend_speedup = pd.merge(speck_backend_speedup_x86, speck_backend_speedup_aarch64, on='version', how='left')
speck_backend_speedup

In [ ]:
def plot_speck_backend_speedup(speedup, filename, fig_height, label):
    with plt.style.context(['science', 'no-latex', 'bright']):
        data = speedup.T.rename(index=backendy_pl)

        fig, ax = plt.subplots(figsize=(FIG_WIDTH, fig_height), layout='constrained')

        sns.heatmap(
            data,
            ax=ax,
            annot=True,
            fmt='.2f',
            cmap='RdYlGn',
            center=1,
            linewidths=1.0,
            cbar_kws={'label': label, 'shrink': 0.85},
        )
        ax.set_xlabel('Wersja Speck')
        ax.set_ylabel('Backend')

        ax.axhline(y=3, color='white', linewidth=2.5)
        ax.axhline(y=3, color='black', linewidth=0.5)

        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
        plt.setp(ax.get_yticklabels(), rotation=0)

        plt.savefig(f'../svg/{filename}', facecolor='white')
        plt.show()

plot_speck_backend_speedup(speck_backend_speedup, 'speck_backend_speedup.svg', FIG_HEIGHT_4, 'Krotność przyspieszenia')

In [ ]:
def parse_turbostat(filepath):
    samples = []
    current = {}

    with open(filepath) as f:
        for line in f:
            parts = line.split()
            if not parts:
                continue

            if parts[0] == "CPU":
                if current:
                    samples.append(current)
                continue

            cpu = "system" if parts[0] == "-" else f"cpu{parts[0]}"
            current[f"{cpu}"] = int(parts[2])
            if len(parts) > 3 and cpu == "system":
                current[f"{cpu}_pkg_watt"] = float(parts[3])

        if current:
            samples.append(current)

    return pd.DataFrame(samples)

In [ ]:
bench_turbostat = parse_turbostat("../data/turbostat-bench.txt")
cpu_cols = sorted(
    [c for c in bench_turbostat.columns if c.startswith("cpu")],
    key=lambda x: int(x.replace("cpu", "").split("_")[0])
)
other_cols = [c for c in bench_turbostat.columns if not c.startswith("cpu")]

bench_turbostat = bench_turbostat[other_cols + cpu_cols]
bench_turbostat

In [ ]:
cols_to_check = cpu_cols + ["system_pkg_watt"]

results = {}
for c in cols_to_check:
    y = bench_turbostat[c].dropna().values
    x = np.arange(len(y))
    slope, intercept, lo, hi = theilslopes(y, x, 0.95)
    results[c] = {"slope_per_sample": slope, "ci_low": lo, "ci_high": hi, "intercept": intercept}

bench_turbostat_trends = pd.DataFrame(results).T
bench_turbostat_trends

In [ ]:
system_turbostat = parse_turbostat("../data/turbostat-system.txt")
cpu_cols = sorted(
    [c for c in system_turbostat.columns if c.startswith("cpu")],
    key=lambda x: int(x.replace("cpu", "").split("_")[0])
)
other_cols = [c for c in system_turbostat.columns if not c.startswith("cpu")]

system_turbostat = system_turbostat[other_cols + cpu_cols]
system_turbostat

In [ ]:
cols_to_check = cpu_cols + ["system_pkg_watt"]

results = {}
for c in cols_to_check:
    y = system_turbostat[c].dropna().values
    x = np.arange(len(y))
    slope, intercept, lo, hi = theilslopes(y, x, 0.95)
    results[c] = {"slope_per_sample": slope, "ci_low": lo, "ci_high": hi, "intercept": intercept}

system_turbostat_trends = pd.DataFrame(results).T
system_turbostat_trends

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


def prepare_data(df, cpu_cols):
    df_freq = df[cpu_cols].copy()
    df_freq["sample"] = np.arange(len(df_freq))
    df_freq_long = df_freq.melt(
        id_vars="sample", var_name="CPU", value_name="freq"
    )
    df_watt = pd.DataFrame({
        "sample": np.arange(len(df)),
        "watt": df["system_pkg_watt"].values,
    })
    return df_freq_long, df_watt


def plot_turbostat(df_raw, trends_df, cpu_cols, filename):
    df_freq_long, df_watt = prepare_data(df_raw, cpu_cols)

    with plt.style.context(['science', 'no-latex', 'bright']):
        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=(FIG_WIDTH / 2, FIG_HEIGHT_3), layout='constrained', sharex=True,
            gridspec_kw={"height_ratios": [2, 1]},
        )

        sns.lineplot(
            data=df_freq_long, x="sample", y="freq", hue="CPU",
            palette=["#4C9BE8"] * len(cpu_cols),
            linewidth=0.7, alpha=0.25, legend=False, ax=ax1,
        )

        median_freq = df_raw[cpu_cols].median(axis=1)
        sns.lineplot(
            x=np.arange(len(median_freq)), y=median_freq.values,
            color="#1F4E79", linewidth=1.6,
            label="Mediana rdzeni", ax=ax1,
        )

        freq_min = trends_df.loc[cpu_cols, "intercept"].min()
        freq_max = trends_df.loc[cpu_cols, "intercept"].max()
        ax1.set_ylim(np.floor(freq_min / 100) * 100, np.ceil(freq_max / 100) * 100)
        ax1.set_ylabel("Częstotliwość [MHz]")
        ax1.set_xlabel("")
        ax1.legend(loc="lower right", frameon=False)

        sns.lineplot(
            data=df_watt, x="sample", y="watt",
            color="#C0392B", linewidth=1.0, ax=ax2,
        )
        ax2.fill_between(
            df_watt["sample"], 0, df_watt["watt"],
            color="#C0392B", alpha=0.12,
        )

        watt_max = df_watt["watt"].max()
        ax2.set_ylim(0, np.ceil(watt_max / 100) * 100)
        ax2.set_ylabel("Pobór mocy [W]")
        ax2.set_xlabel("Numer próbki")

        plt.savefig(f'../svg/{filename}.svg', facecolor='white')
        plt.show()


plot_turbostat(bench_turbostat, bench_turbostat_trends,
               cpu_cols, "turbostat_bench")
plot_turbostat(system_turbostat, system_turbostat_trends,
               cpu_cols, "turbostat_system")

In [ ]:
def print_turbostat_summary(df_raw, trends_df, cpu_cols, label):
    freq_min = trends_df.loc[cpu_cols, "intercept"].min()
    freq_max = trends_df.loc[cpu_cols, "intercept"].max()
    watt_med = trends_df.loc["system_pkg_watt", "intercept"]

    freq_slopes = trends_df.loc[cpu_cols, "slope_per_sample"]
    freq_slope_min = freq_slopes.min()
    freq_slope_max = freq_slopes.max()

    watt_slope = trends_df.loc["system_pkg_watt", "slope_per_sample"]
    watt_ci_low = trends_df.loc["system_pkg_watt", "ci_low"]
    watt_ci_high = trends_df.loc["system_pkg_watt", "ci_high"]

    print(f"=== {label} ===")
    print(f"Częstotliwość:")
    print(f"  Theil–Sen β ∈ [{freq_slope_min:.4f}; {freq_slope_max:.4f}] MHz/próbkę")
    print(f"  Mediany per rdzeń: {freq_min:.0f}–{freq_max:.0f} MHz")
    print(f"Pobór mocy CPU:")
    print(f"  Theil–Sen β = {watt_slope:.4f} W/próbkę  "
          f"(95% CI: [{watt_ci_low:.4f}; {watt_ci_high:.4f}])")
    print(f"  Mediana = {watt_med:.2f} W")
    print()


print_turbostat_summary(bench_turbostat, bench_turbostat_trends,
                        cpu_cols, "Bench")
print_turbostat_summary(system_turbostat, system_turbostat_trends,
                        cpu_cols, "System")

In [ ]:
scalar_turbostat = parse_turbostat("../data/turbostat-search-Scalar.txt")
cpu_cols = sorted(
    [c for c in scalar_turbostat.columns if c.startswith("cpu")],
    key=lambda x: int(x.replace("cpu", "").split("_")[0])
)
other_cols = [c for c in scalar_turbostat.columns if not c.startswith("cpu")]

scalar_turbostat = scalar_turbostat[other_cols + cpu_cols]
scalar_turbostat

In [ ]:
sse2_turbostat = parse_turbostat("../data/turbostat-search-Sse2.txt")
cpu_cols = sorted(
    [c for c in sse2_turbostat.columns if c.startswith("cpu")],
    key=lambda x: int(x.replace("cpu", "").split("_")[0])
)
other_cols = [c for c in sse2_turbostat.columns if not c.startswith("cpu")]

sse2_turbostat = sse2_turbostat[other_cols + cpu_cols]
sse2_turbostat

In [ ]:
avx2_turbostat = parse_turbostat("../data/turbostat-search-Avx2.txt")
cpu_cols = sorted(
    [c for c in avx2_turbostat.columns if c.startswith("cpu")],
    key=lambda x: int(x.replace("cpu", "").split("_")[0])
)
other_cols = [c for c in avx2_turbostat.columns if not c.startswith("cpu")]

avx2_turbostat = avx2_turbostat[other_cols + cpu_cols]
avx2_turbostat

In [ ]:
avx512_turbostat = parse_turbostat("../data/turbostat-search-Avx512.txt")
cpu_cols = sorted(
    [c for c in avx512_turbostat.columns if c.startswith("cpu")],
    key=lambda x: int(x.replace("cpu", "").split("_")[0])
)
other_cols = [c for c in avx512_turbostat.columns if not c.startswith("cpu")]

avx512_turbostat = avx512_turbostat[other_cols + cpu_cols]
avx512_turbostat

In [ ]:
backends = {
    "Scalar": scalar_turbostat,
    "SSE2": sse2_turbostat,
    "AVX2": avx2_turbostat,
    "AVX512": avx512_turbostat,
}

turbostat_compare = pd.concat(
    [
        df[["system", "system_pkg_watt"]]
        .rename(columns={"system": "freq_mhz", "system_pkg_watt": "pkg_watt"})
        .assign(backend=name)
        for name, df in backends.items()
    ],
    ignore_index=True,
)

backend_order = ["Scalar", "SSE2", "AVX2", "AVX512"]

with plt.style.context(['science', 'no-latex', 'bright']):
    fig, ax = plt.subplots(figsize=(FIG_WIDTH / 2, FIG_HEIGHT_4), layout='constrained')
    sns.stripplot(
        data=turbostat_compare, x="backend", y="freq_mhz",
        order=backend_order, hue="backend",
        jitter=0.2, size=9, alpha=0.9, edgecolor="none",
        legend=False, ax=ax,
    )
    freq_min = turbostat_compare["freq_mhz"].min()
    freq_max = turbostat_compare["freq_mhz"].max()
    ax.set_ylim(np.floor(freq_min / 100) * 100, np.ceil(freq_max / 100) * 100)
    ax.set_xlabel("Backend")
    ax.set_ylabel("Częstotliwość Bzy [MHz]")
    plt.savefig('../svg/turbostat_freq_compare.svg', facecolor='white')
    plt.show()

    fig, ax = plt.subplots(figsize=(FIG_WIDTH / 2, FIG_HEIGHT_4), layout='constrained')
    sns.stripplot(
        data=turbostat_compare, x="backend", y="pkg_watt",
        order=backend_order, hue="backend",
        jitter=0.2, size=9, alpha=0.9, edgecolor="none",
        legend=False, ax=ax,
    )
    watt_min = turbostat_compare["pkg_watt"].min()
    watt_max = turbostat_compare["pkg_watt"].max()
    ax.set_ylim(np.floor((watt_min - 10) / 5) * 5, np.ceil((watt_max + 10) / 5) * 5)
    ax.set_xlabel("Backend")
    ax.set_ylabel("Pobór mocy pakietu [W]")
    plt.savefig('../svg/turbostat_power_compare.svg', facecolor='white')
    plt.show()

In [ ]:
f = turbostat_compare["freq_mhz"]
mean = f.mean()
fmin, fmax = f.min(), f.max()
dev_pct = max(abs(fmin - mean), abs(fmax - mean)) / mean * 100

print(f"Częstotliwość między backendami: {fmin:.0f}–{fmax:.0f} MHz "
      f"(rozrzut {fmax - fmin:.0f} MHz)")
print(f"Średnia: {mean:.0f} MHz, maks. odchył od średniej: {dev_pct:.2f}%")

In [ ]:
p = turbostat_compare["pkg_watt"]
mean = p.mean()
pmin, pmax = p.min(), p.max()
dev_pct = max(abs(pmin - mean), abs(pmax - mean)) / mean * 100

print(f"Pobór mocy między backendami: {pmin:.2f}–{pmax:.2f} W "
      f"(rozrzut {pmax - pmin:.2f} W)")
print(f"Średnia: {mean:.2f} W, maks. odchył od średniej: {dev_pct:.2f}%")

In [ ]:
engine = ciretrion[ciretrion['benchmark'] == 'engine']
engine = engine.drop(columns=['benchmark', 'unit', 'throughput_type', 'function'])
engine['time_per_iter_ns'] = engine['sample_measured_value'] / engine['iteration_count']

engine['keys_per_sec'] = (
        engine['throughput_num'] / (engine['time_per_iter_ns'] / 1e9)
)
engine

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import bootstrap

group_cols = ['backend', 'architecture', 'version', 'suffix']


def analyze_group(group):
    t = (group['sample_measured_value'] / group['iteration_count']).to_numpy()
    n_keys = group['throughput_num'].iloc[0]

    median_ns = np.median(t)
    mad_ns = np.median(np.abs(t - median_ns))
    mad_sigma = 1.4826 * mad_ns

    res_median = bootstrap(
        (t,), np.median,
        n_resamples=10_000,
        confidence_level=0.95,
        method='BCa',
        random_state=0,
    )
    ci_low, ci_high = res_median.confidence_interval

    mkeys_per_sec = n_keys / (median_ns / 1e9) / 1e6
    mkeys_low = n_keys / (ci_high / 1e9) / 1e6
    mkeys_high = n_keys / (ci_low / 1e9) / 1e6

    rel_mad = mad_ns / median_ns
    p05, p95 = np.percentile(t, [5, 95])
    spread = (p95 - p05) / median_ns

    return pd.Series({
        'n_samples': len(t),
        'median_ns': median_ns,
        'ci95_low_ns': ci_low,
        'ci95_high_ns': ci_high,
        'mad_ns': mad_ns,
        'mad_sigma_ns': mad_sigma,
        'rel_mad': rel_mad,
        'spread_p05_p95': spread,
        'Mkeys_per_sec': mkeys_per_sec,
        'Mkeys_per_sec_ci_low': mkeys_low,
        'Mkeys_per_sec_ci_high': mkeys_high,
    })


engine_stats = (
    engine
    .groupby(group_cols)
    .apply(analyze_group, include_groups=False)
    .reset_index()
)
print(engine_stats)

In [ ]:
def make_engine_pivot(df, backend_order):
    pivot = (
        df.pivot_table(
            index=['backend', 'version'],
            columns='suffix',
            values='Mkeys_per_sec',
        )
        .rename(columns={1.0: '1', 2.0: '2', 3.0: '3'})
    ).round(2).reset_index()
    pivot.columns.name = None

    version_order = ['32_64', '48_72', '48_96', '64_96', '64_128', '96_96', '96_144', '128_128', '128_192', '128_256']
    pivot['backend'] = pd.Categorical(pivot['backend'], categories=backend_order, ordered=True)
    pivot['version'] = pd.Categorical(pivot['version'], categories=version_order, ordered=True)
    return pivot.sort_values(['backend', 'version']).reset_index(drop=True)

engine_pivot_x86 = make_engine_pivot(
    engine_stats[engine_stats['architecture'] == 'x86_64'],
    ['scalar', 'sse2', 'avx2', 'avx512']
)
engine_pivot_aarch64 = make_engine_pivot(
    engine_stats[engine_stats['architecture'] == 'aarch64'],
    ['scalar', 'neon']
)

engine_pivot_x86

In [ ]:
for pivot, name in [(engine_pivot_x86, 'x86'), (engine_pivot_aarch64, 'aarch64')]:
    out = pivot.copy()
    out['version'] = out['version'].astype(str).str.replace('_', '/', regex=False)
    out.to_csv(f'../csv/engine_pivot_{name}.csv', index=False)

In [ ]:
def make_engine_pivot_mean(pivot, backend_order):
    mean = pivot.groupby('backend')[['1', '2', '3']].agg(gmean).reset_index()
    mean['backend'] = pd.Categorical(mean['backend'], categories=backend_order, ordered=True)
    mean = mean.sort_values('backend')
    return mean.set_index('backend')[['1', '2', '3']]

engine_pivot_mean_x86 = make_engine_pivot_mean(engine_pivot_x86, ['scalar', 'sse2', 'avx2', 'avx512'])
engine_pivot_mean_aarch64 = make_engine_pivot_mean(engine_pivot_aarch64, ['scalar', 'neon'])

engine_pivot_mean_x86

In [ ]:
def plot_engine_heatmap(pivot_mean, filename, width):
    data = pivot_mean.T.rename(columns=backendy_pl)

    with plt.style.context(['science', 'no-latex', 'bright']):
        fig, ax = plt.subplots(figsize=(width, FIG_HEIGHT_7), layout='constrained')

        sns.heatmap(
            data,
            ax=ax,
            cmap='RdYlGn',
            center=0,
            annot=True,
            fmt='.2f',
            linewidths=1.0,
            cbar_kws={'label': 'Przepustowość\n[Mkeys/s]', 'shrink': 0.85},
        )

        ax.set_xlabel('Backend')
        ax.set_ylabel('Sufiks')

        plt.setp(ax.get_yticklabels(), rotation=0)

        plt.savefig(f'../svg/{filename}', facecolor='white')
        plt.show()

plot_engine_heatmap(engine_pivot_mean_x86, 'engine_pivot_backend_heatmap_x86.svg', FIG_WIDTH * 0.615)
plot_engine_heatmap(engine_pivot_mean_aarch64, 'engine_pivot_backend_heatmap_aarch64.svg', FIG_WIDTH * 0.385)

In [ ]:
def make_engine_single_backend_pivot(df, architecture, backend):
    version_order = ['32_64', '48_72', '48_96', '64_96', '64_128',
                     '96_96', '96_144', '128_128', '128_192', '128_256']

    pivot = (
        df[(df['architecture'] == architecture) & (df['backend'] == backend)]
        .pivot_table(
            index=['backend', 'version'],
            columns='suffix',
            values='Mkeys_per_sec',
        )
        .rename(columns={1.0: '1', 2.0: '2', 3.0: '3'})
    ).round(2).reset_index()

    pivot.columns.name = None
    pivot['version'] = pd.Categorical(pivot['version'], categories=version_order, ordered=True)
    pivot = pivot.sort_values('version').reset_index(drop=True)
    pivot['version'] = pivot['version'].astype(str).str.replace('_', '/', regex=False)
    return pivot.set_index('version')[['1', '2', '3']]

engine_pivot_avx512 = make_engine_single_backend_pivot(engine_stats, 'x86_64', 'avx512')
engine_pivot_neon   = make_engine_single_backend_pivot(engine_stats, 'aarch64', 'neon')

print(engine_pivot_avx512)


In [ ]:
def plot_engine_single_backend_heatmap(pivot, filename):
    with plt.style.context(['science', 'no-latex', 'bright']):
        fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT_5), layout='constrained')

        sns.heatmap(
            pivot.T,
            ax=ax,
            cmap='RdYlGn',
            center=0,
            annot=True,
            fmt='.1f',
            linewidths=1.0,
            cbar_kws={'label': 'Przepustowość\n[Mkeys/s]', 'shrink': 0.85},
        )

        ax.set_xlabel('Backend')
        ax.set_ylabel('Wartość sufiksu')

        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
        plt.setp(ax.get_yticklabels(), rotation=0)

        plt.savefig(f'../svg/{filename}', facecolor='white')
        plt.show()

plot_engine_single_backend_heatmap(engine_pivot_avx512, 'engine_pivot_backend_avx512_heatmap.svg')
plot_engine_single_backend_heatmap(engine_pivot_neon,   'engine_pivot_backend_neon_heatmap.svg')


In [ ]:
def parse_pm(filepath):
    samples = []
    current = {}

    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line.startswith("*** Sampled system activity"):
                if current and any(k.startswith("cpu") for k in current):
                    samples.append(current)
                current = {}
                continue
            if line.startswith("CPU ") and "frequency:" in line:
                parts = line.split()
                current[f"cpu{int(parts[1])}"] = int(parts[3])
                continue
            if line.startswith("CPU Power:"):
                current["system_pkg_watt"] = float(line.split()[2]) / 1000.0

    if current and any(k.startswith("cpu") for k in current):
        samples.append(current)

    df = pd.DataFrame(samples)
    cpu_cols_local = sorted(
        [c for c in df.columns if c.startswith("cpu")],
        key=lambda x: int(x.replace("cpu", ""))
    )
    df["system"] = df[cpu_cols_local].mean(axis=1)
    return df


In [ ]:
bench_pm = parse_pm("../data/pm-bench.txt")
cpu_cols = sorted(
    [c for c in bench_pm.columns if c.startswith("cpu")],
    key=lambda x: int(x.replace("cpu", ""))
)
other_cols = [c for c in bench_pm.columns if not c.startswith("cpu")]

bench_pm = bench_pm[other_cols + cpu_cols]
bench_pm


In [ ]:
cols_to_check = cpu_cols + ["system_pkg_watt"]

results = {}
for c in cols_to_check:
    y = bench_pm[c].dropna().values
    x = np.arange(len(y))
    slope, intercept, lo, hi = theilslopes(y, x, 0.95)
    results[c] = {"slope_per_sample": slope, "ci_low": lo, "ci_high": hi, "intercept": intercept}

bench_pm_trends = pd.DataFrame(results).T
bench_pm_trends


In [ ]:
system_pm = parse_pm("../data/pm-system.txt")
cpu_cols = sorted(
    [c for c in system_pm.columns if c.startswith("cpu")],
    key=lambda x: int(x.replace("cpu", ""))
)
other_cols = [c for c in system_pm.columns if not c.startswith("cpu")]

system_pm = system_pm[other_cols + cpu_cols]
system_pm


In [ ]:
cols_to_check = cpu_cols + ["system_pkg_watt"]

results = {}
for c in cols_to_check:
    y = system_pm[c].dropna().values
    x = np.arange(len(y))
    slope, intercept, lo, hi = theilslopes(y, x, 0.95)
    results[c] = {"slope_per_sample": slope, "ci_low": lo, "ci_high": hi, "intercept": intercept}

system_pm_trends = pd.DataFrame(results).T
system_pm_trends


In [ ]:
def plot_pm(df_raw, trends_df, cpu_cols, filename):
    df_freq_long, df_watt = prepare_data(df_raw, cpu_cols)

    with plt.style.context(["science", "no-latex", "bright"]):
        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=(FIG_WIDTH, FIG_HEIGHT_3), layout="constrained", sharex=True,
            gridspec_kw={"height_ratios": [2, 1]},
        )

        sns.lineplot(
            data=df_freq_long, x="sample", y="freq", hue="CPU",
            palette=["#4C9BE8"] * len(cpu_cols),
            linewidth=0.3, alpha=0.25, legend=False, ax=ax1,
        )

        median_freq = df_raw[cpu_cols].median(axis=1)
        sns.lineplot(
            x=np.arange(len(median_freq)), y=median_freq.values,
            color="#1F4E79", linewidth=0.5,
            label="Mediana rdzeni", ax=ax1,
        )

        freq_min = trends_df.loc[cpu_cols, "intercept"].min()
        freq_max = trends_df.loc[cpu_cols, "intercept"].max()
        ax1.set_ylim(np.floor(freq_min / 100) * 100, np.ceil(freq_max / 100) * 100)
        ax1.set_ylabel("Częstotliwość [MHz]")
        ax1.set_xlabel("")
        ax1.legend(loc="lower right", frameon=False)

        sns.lineplot(
            data=df_watt, x="sample", y="watt",
            color="#C0392B", linewidth=0.5, ax=ax2,
        )
        ax2.fill_between(
            df_watt["sample"], 0, df_watt["watt"],
            color="#C0392B", alpha=0.12,
        )

        watt_max = df_watt["watt"].max()
        ax2.set_ylim(0, np.ceil(watt_max * 10) / 10)
        ax2.set_ylabel("Pobór mocy CPU [W]")
        ax2.set_xlabel("Numer próbki")

        plt.savefig(f"../svg/{filename}.svg", facecolor="white")
        plt.show()


plot_pm(bench_pm, bench_pm_trends, cpu_cols, "pm_bench")
plot_pm(system_pm, system_pm_trends, cpu_cols, "pm_system")


In [ ]:
def print_pm_summary(df_raw, trends_df, cpu_cols, label):
    freq_min = trends_df.loc[cpu_cols, "intercept"].min()
    freq_max = trends_df.loc[cpu_cols, "intercept"].max()
    watt_med = trends_df.loc["system_pkg_watt", "intercept"]

    freq_slopes = trends_df.loc[cpu_cols, "slope_per_sample"]
    freq_slope_min = freq_slopes.min()
    freq_slope_max = freq_slopes.max()

    watt_slope = trends_df.loc["system_pkg_watt", "slope_per_sample"]
    watt_ci_low = trends_df.loc["system_pkg_watt", "ci_low"]
    watt_ci_high = trends_df.loc["system_pkg_watt", "ci_high"]

    print(f"=== {label} ===")
    print(f"Częstotliwość:")
    print(f"  Theil–Sen β ∈ [{freq_slope_min:.4f}; {freq_slope_max:.4f}] MHz/próbkę")
    print(f"  Mediany per rdzeń: {freq_min:.0f}–{freq_max:.0f} MHz")
    print(f"Pobór mocy CPU:")
    print(f"  Theil–Sen β = {watt_slope:.4f} W/próbkę  "
          f"(95% CI: [{watt_ci_low:.4f}; {watt_ci_high:.4f}])")
    print(f"  Mediana = {watt_med:.3f} W")
    print()


print_pm_summary(bench_pm, bench_pm_trends, cpu_cols, "Bench")
print_pm_summary(system_pm, system_pm_trends, cpu_cols, "System")


In [ ]:
scalar_pm = parse_pm("../data/pm-search-Scalar.txt")
cpu_cols = sorted(
    [c for c in scalar_pm.columns if c.startswith("cpu")],
    key=lambda x: int(x.replace("cpu", ""))
)
other_cols = [c for c in scalar_pm.columns if not c.startswith("cpu")]

scalar_pm = scalar_pm[other_cols + cpu_cols]
scalar_pm


In [ ]:
neon_pm = parse_pm("../data/pm-search-Neon.txt")
cpu_cols = sorted(
    [c for c in neon_pm.columns if c.startswith("cpu")],
    key=lambda x: int(x.replace("cpu", ""))
)
other_cols = [c for c in neon_pm.columns if not c.startswith("cpu")]

neon_pm = neon_pm[other_cols + cpu_cols]
neon_pm


In [ ]:
backends = {
    "Scalar": scalar_pm,
    "NEON": neon_pm,
}

pm_compare = pd.concat(
    [
        df[["system", "system_pkg_watt"]]
        .rename(columns={"system": "freq_mhz", "system_pkg_watt": "pkg_watt"})
        .assign(backend=name)
        for name, df in backends.items()
    ],
    ignore_index=True,
)

backend_order = ["Scalar", "NEON"]

with plt.style.context(["science", "no-latex", "bright"]):
    fig, ax = plt.subplots(figsize=(FIG_WIDTH / 2, FIG_HEIGHT_4), layout="constrained")
    sns.stripplot(
        data=pm_compare, x="backend", y="freq_mhz",
        order=backend_order, hue="backend",
        jitter=0.2, size=9, alpha=0.9, edgecolor="none",
        legend=False, ax=ax,
    )
    freq_min = pm_compare["freq_mhz"].min()
    freq_max = pm_compare["freq_mhz"].max()
    ax.set_ylim(np.floor(freq_min / 100) * 100, np.ceil(freq_max / 100) * 100)
    ax.set_xlabel("Backend")
    ax.set_ylabel("Częstotliwość [MHz]")
    plt.savefig("../svg/pm_freq_compare.svg", facecolor="white")
    plt.show()

    fig, ax = plt.subplots(figsize=(FIG_WIDTH / 2, FIG_HEIGHT_4), layout="constrained")
    sns.stripplot(
        data=pm_compare, x="backend", y="pkg_watt",
        order=backend_order, hue="backend",
        jitter=0.2, size=9, alpha=0.9, edgecolor="none",
        legend=False, ax=ax,
    )
    watt_min = pm_compare["pkg_watt"].min()
    watt_max = pm_compare["pkg_watt"].max()
    ax.set_ylim(0, np.ceil(watt_max * 10) / 10)
    ax.set_xlabel("Backend")
    ax.set_ylabel("Pobór mocy CPU [W]")
    plt.savefig("../svg/pm_power_compare.svg", facecolor="white")
    plt.show()


In [ ]:
f = pm_compare["freq_mhz"]
mean = f.mean()
fmin, fmax = f.min(), f.max()
dev_pct = max(abs(fmin - mean), abs(fmax - mean)) / mean * 100

print(f"Częstotliwość między backendami: {fmin:.0f}–{fmax:.0f} MHz "
      f"(rozrzut {fmax - fmin:.0f} MHz)")
print(f"Średnia: {mean:.0f} MHz, maks. odchył od średniej: {dev_pct:.2f}%")


In [ ]:
p = pm_compare["pkg_watt"]
mean = p.mean()
pmin, pmax = p.min(), p.max()
dev_pct = max(abs(pmin - mean), abs(pmax - mean)) / mean * 100

print(f"Pobór mocy między backendami: {pmin:.3f}–{pmax:.3f} W "
      f"(rozrzut {pmax - pmin:.3f} W)")
print(f"Średnia: {mean:.3f} W, maks. odchył od średniej: {dev_pct:.2f}%")
